In [210]:
# Selecciona Balance y Período para descargar: 
BalanceFile = "Balance_2025.csv"
Periodo = 'Anual' # 'Anual' si es Balance del Año. 'Q1'..'Q4' si es Trimestral.

### Tranformación Automatizada Datos Holded:

In [211]:
import pandas as pd 
import numpy as np 

In [212]:
# Importamos data:
    # BalanceFile = "Balance2022.csv"
    # Periodo = 'Anual'

Balance = pd.read_csv(f"C:/Users/Jorge Lozano/Desktop/GitHub_Holded/Data/Balance/{BalanceFile}", header = None, sep = ',', usecols = [0, 1]) 


In [213]:
# Renombramos nombres de columnas:
Balance = Balance.rename(columns={0 : 'Subcuenta', 1 : 'Valor'})
# Borrar filas basura sin Subcuenta (ej. la fila 'Total' del encabezado del CSV):
Balance = Balance[Balance['Subcuenta'].notna()]

* 1. Partida:

In [214]:
# Creamos columna con el nombre de la partida:
Balance['Partida'] = Balance['Subcuenta'].where(Balance['Valor'].isna()).ffill()
# Borrar fila original donde Valor es NaN:
Balance = Balance[~Balance['Valor'].isna()]

In [215]:
Balance

,Subcuenta,Valor,Partida
6,Activo No Corriente,"123137,81",ACTIVO
7,I. Inmovilizado intangible,"55534,05",ACTIVO
8,5. Aplicaciones informáticas,"55534,05",ACTIVO
9,206 Aplicaciones informáticas,718791,ACTIVO
10,20600001 - CASH LESS DESARROLLO SOFTWARE,718791,ACTIVO
...,...,...,...
222,47510000 - HP ACREED RETEN PROFESIONAL,"1002,40",PASIVO
223,47510001 - HP ACREED RETEN TRABAJADOR,"62189,84",PASIVO
224,476 Organismos de la Seguridad Social,acreedores,PASIVO
225,47600000 - ORGANIDE LA SSSSACREEDORES,"23553,70",PASIVO


* 2. Subpartida:

In [216]:
Subpartidas = pd.Series(['Activo No Corriente', 'Activo Corriente', 'Patrimonio Neto',
                       'Pasivo Corriente', 'Pasivo No Corriente'])
print(Subpartidas)

0    Activo No Corriente
1       Activo Corriente
2        Patrimonio Neto
3       Pasivo Corriente
4    Pasivo No Corriente
dtype: object


In [217]:
# 1. Creamos columna para filas con Subpartidas:
Balance['Subpartida'] = Balance['Subcuenta'].where(Balance['Subcuenta'].isin(Subpartidas)).ffill()
# 2. Eliminamos la columna con Subpartida Original:
Balance = Balance[~Balance['Subcuenta'].isin(Subpartidas)]

In [218]:
Balance

,Subcuenta,Valor,Partida,Subpartida
7,I. Inmovilizado intangible,"55534,05",ACTIVO,Activo No Corriente
8,5. Aplicaciones informáticas,"55534,05",ACTIVO,Activo No Corriente
9,206 Aplicaciones informáticas,718791,ACTIVO,Activo No Corriente
10,20600001 - CASH LESS DESARROLLO SOFTWARE,718791,ACTIVO,Activo No Corriente
11,2806 Amortización acumulada de aplicaciones in...,"-663256,75",ACTIVO,Activo No Corriente
...,...,...,...,...
222,47510000 - HP ACREED RETEN PROFESIONAL,"1002,40",PASIVO,Pasivo Corriente
223,47510001 - HP ACREED RETEN TRABAJADOR,"62189,84",PASIVO,Pasivo Corriente
224,476 Organismos de la Seguridad Social,acreedores,PASIVO,Pasivo Corriente
225,47600000 - ORGANIDE LA SSSSACREEDORES,"23553,70",PASIVO,Pasivo Corriente


* 3. Grupo:

In [219]:
Grupo_id = pd.Series(['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X'])

In [220]:
Grupo_id = Grupo_id.tolist()

In [221]:
# Para crear columna Grupo:
Balance['Grupo'] = Balance['Subcuenta'].where(Balance['Subcuenta'].str.startswith(tuple(Grupo_id))).ffill()
# Para Borrar Filas Grupo Original:
Balance = Balance[~Balance['Subcuenta'].str.startswith(tuple(Grupo_id))]

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_12892\4050577939.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Balance['Grupo'] = Balance['Subcuenta'].where(Balance['Subcuenta'].str.startswith(tuple(Grupo_id))).ffill()


In [222]:
Balance

,Subcuenta,Valor,Partida,Subpartida,Grupo
8,5. Aplicaciones informáticas,"55534,05",ACTIVO,Activo No Corriente,I. Inmovilizado intangible
9,206 Aplicaciones informáticas,718791,ACTIVO,Activo No Corriente,I. Inmovilizado intangible
10,20600001 - CASH LESS DESARROLLO SOFTWARE,718791,ACTIVO,Activo No Corriente,I. Inmovilizado intangible
11,2806 Amortización acumulada de aplicaciones in...,"-663256,75",ACTIVO,Activo No Corriente,I. Inmovilizado intangible
12,28060001 - AMORTACUMCASH LESS,"-663256,75",ACTIVO,Activo No Corriente,I. Inmovilizado intangible
...,...,...,...,...,...
222,47510000 - HP ACREED RETEN PROFESIONAL,"1002,40",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar
223,47510001 - HP ACREED RETEN TRABAJADOR,"62189,84",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar
224,476 Organismos de la Seguridad Social,acreedores,PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar
225,47600000 - ORGANIDE LA SSSSACREEDORES,"23553,70",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar


* Subgrupo:

In [223]:
Subgrupo_id = pd.Series(['1.', '2.', '3.', '4.', '5.', '6.', '7.',
                         '8.', '9.']).tolist()

In [224]:
# Creamos columna Subgrupo:
Balance['SubGrupo'] = Balance['Subcuenta'].where(Balance['Subcuenta'].str.startswith(tuple(Subgrupo_id))).ffill()
# Eliminamos columna original con Subgrupo:
Balance = Balance[~Balance['Subcuenta'].str.startswith(tuple(Subgrupo_id))]

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_12892\3592115905.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Balance['SubGrupo'] = Balance['Subcuenta'].where(Balance['Subcuenta'].str.startswith(tuple(Subgrupo_id))).ffill()


In [225]:
Balance

,Subcuenta,Valor,Partida,Subpartida,Grupo,SubGrupo
9,206 Aplicaciones informáticas,718791,ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas
10,20600001 - CASH LESS DESARROLLO SOFTWARE,718791,ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas
11,2806 Amortización acumulada de aplicaciones in...,"-663256,75",ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas
12,28060001 - AMORTACUMCASH LESS,"-663256,75",ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas
15,216 Mobiliario,"21261,56",ACTIVO,Activo No Corriente,II. Inmovilizado material,2. Instalaciones técnicas y otro inmovilizado ...
...,...,...,...,...,...,...
222,47510000 - HP ACREED RETEN PROFESIONAL,"1002,40",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas
223,47510001 - HP ACREED RETEN TRABAJADOR,"62189,84",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas
224,476 Organismos de la Seguridad Social,acreedores,PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas
225,47600000 - ORGANIDE LA SSSSACREEDORES,"23553,70",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas


* Cuenta:

In [226]:
print(Balance)

                                             Subcuenta       Valor Partida  \
9                        206 Aplicaciones informáticas      718791  ACTIVO   
10            20600001 - CASH LESS DESARROLLO SOFTWARE      718791  ACTIVO   
11   2806 Amortización acumulada de aplicaciones in...  -663256,75  ACTIVO   
12                       28060001 - AMORTACUMCASH LESS  -663256,75  ACTIVO   
15                                      216 Mobiliario    21261,56  ACTIVO   
..                                                 ...         ...     ...   
222             47510000 - HP ACREED RETEN PROFESIONAL     1002,40  PASIVO   
223              47510001 - HP ACREED RETEN TRABAJADOR    62189,84  PASIVO   
224              476 Organismos de la Seguridad Social  acreedores  PASIVO   
225              47600000 - ORGANIDE LA SSSSACREEDORES    23553,70  PASIVO   
227                     Total Patrimonio Neto y Pasivo   2551146,5  PASIVO   

              Subpartida                                       

In [227]:
Balance['Subcuenta'].where(pd.to_numeric(Balance['Subcuenta'].str.extract(r'^(\d+)')[0]) < 1000).ffill()

9              206 Aplicaciones informáticas
10             206 Aplicaciones informáticas
11             206 Aplicaciones informáticas
12             206 Aplicaciones informáticas
15                            216 Mobiliario
                       ...                  
222    465 Remuneraciones pendientes de pago
223    465 Remuneraciones pendientes de pago
224    476 Organismos de la Seguridad Social
225    476 Organismos de la Seguridad Social
227    476 Organismos de la Seguridad Social
Name: Subcuenta, Length: 175, dtype: object

In [228]:
Balance['Cuenta'] = Balance['Subcuenta'].where(pd.to_numeric(Balance['Subcuenta'].str.extract(r'^(\d+)')[0]) < 10000).ffill()

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_12892\2638717216.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Balance['Cuenta'] = Balance['Subcuenta'].where(pd.to_numeric(Balance['Subcuenta'].str.extract(r'^(\d+)')[0]) < 10000).ffill()


In [229]:
Balance

,Subcuenta,Valor,Partida,Subpartida,Grupo,SubGrupo,Cuenta
9,206 Aplicaciones informáticas,718791,ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas,206 Aplicaciones informáticas
10,20600001 - CASH LESS DESARROLLO SOFTWARE,718791,ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas,206 Aplicaciones informáticas
11,2806 Amortización acumulada de aplicaciones in...,"-663256,75",ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas,2806 Amortización acumulada de aplicaciones in...
12,28060001 - AMORTACUMCASH LESS,"-663256,75",ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas,2806 Amortización acumulada de aplicaciones in...
15,216 Mobiliario,"21261,56",ACTIVO,Activo No Corriente,II. Inmovilizado material,2. Instalaciones técnicas y otro inmovilizado ...,216 Mobiliario
...,...,...,...,...,...,...,...
222,47510000 - HP ACREED RETEN PROFESIONAL,"1002,40",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,4751 Hacienda Pública
223,47510001 - HP ACREED RETEN TRABAJADOR,"62189,84",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,4751 Hacienda Pública
224,476 Organismos de la Seguridad Social,acreedores,PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,476 Organismos de la Seguridad Social
225,47600000 - ORGANIDE LA SSSSACREEDORES,"23553,70",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,476 Organismos de la Seguridad Social


In [230]:
Balance['Cuenta'] = Balance['Cuenta'].where(pd.to_numeric(Balance['Subcuenta'].str.extract(r'^(\d+)')[0]) < 10000).ffill()

Balance['Cuenta'].where(pd.to_numeric(Balance['Subcuenta'].str.extract(r'^(\d+)')[0]) < 10000)

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_12892\22515602.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Balance['Cuenta'] = Balance['Cuenta'].where(pd.to_numeric(Balance['Subcuenta'].str.extract(r'^(\d+)')[0]) < 10000).ffill()


9                          206 Aplicaciones informáticas
10                                                   NaN
11     2806 Amortización acumulada de aplicaciones in...
12                                                   NaN
15                                        216 Mobiliario
                             ...                        
222                                                  NaN
223                                                  NaN
224                476 Organismos de la Seguridad Social
225                                                  NaN
227                                                  NaN
Name: Cuenta, Length: 175, dtype: object

In [231]:
Balance = Balance[~(pd.to_numeric(Balance['Subcuenta'].str.extract(r'^(\d+)')[0]) < 10000)]

In [232]:
Balance

,Subcuenta,Valor,Partida,Subpartida,Grupo,SubGrupo,Cuenta
10,20600001 - CASH LESS DESARROLLO SOFTWARE,718791,ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas,206 Aplicaciones informáticas
12,28060001 - AMORTACUMCASH LESS,"-663256,75",ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas,2806 Amortización acumulada de aplicaciones in...
16,21600001 - 5 ASIENTOS RECLINABLES,1650,ACTIVO,Activo No Corriente,II. Inmovilizado material,2. Instalaciones técnicas y otro inmovilizado ...,216 Mobiliario
17,21600002 - 5 MESAS OFICINA LINEAL,1061,ACTIVO,Activo No Corriente,II. Inmovilizado material,2. Instalaciones técnicas y otro inmovilizado ...,216 Mobiliario
18,21600003 - 5 CAJONERAS RODANTES,956,ACTIVO,Activo No Corriente,II. Inmovilizado material,2. Instalaciones técnicas y otro inmovilizado ...,216 Mobiliario
...,...,...,...,...,...,...,...
220,47500000 - HACIENDA PÚBLICA ACREEDORA PO,"60818,41",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,4750 Hacienda Pública
222,47510000 - HP ACREED RETEN PROFESIONAL,"1002,40",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,4751 Hacienda Pública
223,47510001 - HP ACREED RETEN TRABAJADOR,"62189,84",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,4751 Hacienda Pública
225,47600000 - ORGANIDE LA SSSSACREEDORES,"23553,70",PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,476 Organismos de la Seguridad Social


In [233]:
# Configuración Año y Periodo (Establecidos en el inicio del script)

Año = BalanceFile[8:12]

Balance['Año'] = Año
Balance['Periodo'] = Periodo

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_12892\919494026.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Balance['Año'] = Año
C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_12892\919494026.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Balance['Periodo'] = Periodo


* Ordenar Columnas:

In [234]:
Balance = Balance[['Año', 'Periodo', 'Partida', 'Subpartida', 'Grupo', 'SubGrupo', 'Cuenta', 'Subcuenta', 'Valor']]

In [235]:
Balance

,Año,Periodo,Partida,Subpartida,Grupo,SubGrupo,Cuenta,Subcuenta,Valor
10,2025,Anual,ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas,206 Aplicaciones informáticas,20600001 - CASH LESS DESARROLLO SOFTWARE,718791
12,2025,Anual,ACTIVO,Activo No Corriente,I. Inmovilizado intangible,5. Aplicaciones informáticas,2806 Amortización acumulada de aplicaciones in...,28060001 - AMORTACUMCASH LESS,"-663256,75"
16,2025,Anual,ACTIVO,Activo No Corriente,II. Inmovilizado material,2. Instalaciones técnicas y otro inmovilizado ...,216 Mobiliario,21600001 - 5 ASIENTOS RECLINABLES,1650
17,2025,Anual,ACTIVO,Activo No Corriente,II. Inmovilizado material,2. Instalaciones técnicas y otro inmovilizado ...,216 Mobiliario,21600002 - 5 MESAS OFICINA LINEAL,1061
18,2025,Anual,ACTIVO,Activo No Corriente,II. Inmovilizado material,2. Instalaciones técnicas y otro inmovilizado ...,216 Mobiliario,21600003 - 5 CAJONERAS RODANTES,956
...,...,...,...,...,...,...,...,...,...
220,2025,Anual,PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,4750 Hacienda Pública,47500000 - HACIENDA PÚBLICA ACREEDORA PO,"60818,41"
222,2025,Anual,PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,4751 Hacienda Pública,47510000 - HP ACREED RETEN PROFESIONAL,"1002,40"
223,2025,Anual,PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,4751 Hacienda Pública,47510001 - HP ACREED RETEN TRABAJADOR,"62189,84"
225,2025,Anual,PASIVO,Pasivo Corriente,V. Acreedores comerciales y otras cuentas a pagar,6. Otras deudas con las Administraciones Públicas,476 Organismos de la Seguridad Social,47600000 - ORGANIDE LA SSSSACREEDORES,"23553,70"


* Pendiente Transformación: Desglosar Columnas correspondientes en {Id y Nombre}

In [236]:
# Division Columnas y creación columnas_id's: 

Balance[['Grupo_id', 'Grupo']] = Balance['Grupo'].str.split('. ', n = 1, expand = True)              # Grupo
Balance[['SubGrupo_id', 'SubGrupo']] = Balance['SubGrupo'].str.split('. ', n = 1, expand = True)     # SubGrupo
Balance[['Cuenta_id', 'Cuenta']] = Balance['Cuenta'].str.split(' ', n = 1, expand = True)            # Cuenta
Balance[['Subcuenta_id', 'Subcuenta']] = Balance['Subcuenta'].str.split(' - ', n = 1, expand = True) # Subcuenta

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_12892\4237960539.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Balance[['Grupo_id', 'Grupo']] = Balance['Grupo'].str.split('. ', n = 1, expand = True)              # Grupo
C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_12892\4237960539.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Balance[['SubGrupo_id', 'SubGrupo']] = Balance['SubGrupo'].str.split('. ', n = 1, expand = True)     # SubGrupo
C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_

In [237]:
Balance.head(5)

,Año,Periodo,Partida,Subpartida,Grupo,SubGrupo,Cuenta,Subcuenta,Valor,Grupo_id,SubGrupo_id,Cuenta_id,Subcuenta_id
10,2025,Anual,ACTIVO,Activo No Corriente,Inmovilizado intangible,Aplicaciones informáticas,Aplicaciones informáticas,CASH LESS DESARROLLO SOFTWARE,718791,I,5,206,20600001
12,2025,Anual,ACTIVO,Activo No Corriente,Inmovilizado intangible,Aplicaciones informáticas,Amortización acumulada de aplicaciones informá...,AMORTACUMCASH LESS,"-663256,75",I,5,2806,28060001
16,2025,Anual,ACTIVO,Activo No Corriente,Inmovilizado material,Instalaciones técnicas y otro inmovilizado mat...,Mobiliario,5 ASIENTOS RECLINABLES,1650,II,2,216,21600001
17,2025,Anual,ACTIVO,Activo No Corriente,Inmovilizado material,Instalaciones técnicas y otro inmovilizado mat...,Mobiliario,5 MESAS OFICINA LINEAL,1061,II,2,216,21600002
18,2025,Anual,ACTIVO,Activo No Corriente,Inmovilizado material,Instalaciones técnicas y otro inmovilizado mat...,Mobiliario,5 CAJONERAS RODANTES,956,II,2,216,21600003


In [238]:
Balance.columns

Index(['Año', 'Periodo', 'Partida', 'Subpartida', 'Grupo', 'SubGrupo',
       'Cuenta', 'Subcuenta', 'Valor', 'Grupo_id', 'SubGrupo_id', 'Cuenta_id',
       'Subcuenta_id'],
      dtype='object')

In [239]:
# Ordenamos el dataframe: 
Balance = Balance[['Año', 'Periodo', 'Partida', 'Subpartida', 'Grupo_id', 'Grupo', 'SubGrupo_id', 'SubGrupo',
                 'Cuenta_id', 'Cuenta', 'Subcuenta_id', 'Subcuenta', 'Valor']]

In [240]:
# Cambiamos tipo de dato de Valor de Object a Numeric:
Balance['Valor'] = Balance['Valor'].str.replace('.', '', regex=False)   # Elimina los puntos que separan miles
Balance['Valor'] = Balance['Valor'].str.replace(',', '.', regex=False)  # Reemplaza las comas por puntos decimales
Balance['Valor'] = pd.to_numeric(Balance['Valor'], errors='coerce')

In [241]:
Balance.head(10)

,Año,Periodo,Partida,Subpartida,Grupo_id,Grupo,SubGrupo_id,SubGrupo,Cuenta_id,Cuenta,Subcuenta_id,Subcuenta,Valor
10,2025,Anual,ACTIVO,Activo No Corriente,I,Inmovilizado intangible,5,Aplicaciones informáticas,206,Aplicaciones informáticas,20600001,CASH LESS DESARROLLO SOFTWARE,718791.00
12,2025,Anual,ACTIVO,Activo No Corriente,I,Inmovilizado intangible,5,Aplicaciones informáticas,2806,Amortización acumulada de aplicaciones informá...,28060001,AMORTACUMCASH LESS,-663256.75
16,2025,Anual,ACTIVO,Activo No Corriente,II,Inmovilizado material,2,Instalaciones técnicas y otro inmovilizado mat...,216,Mobiliario,21600001,5 ASIENTOS RECLINABLES,1650.00
17,2025,Anual,ACTIVO,Activo No Corriente,II,Inmovilizado material,2,Instalaciones técnicas y otro inmovilizado mat...,216,Mobiliario,21600002,5 MESAS OFICINA LINEAL,1061.00
18,2025,Anual,ACTIVO,Activo No Corriente,II,Inmovilizado material,2,Instalaciones técnicas y otro inmovilizado mat...,216,Mobiliario,21600003,5 CAJONERAS RODANTES,956.00
19,2025,Anual,ACTIVO,Activo No Corriente,II,Inmovilizado material,2,Instalaciones técnicas y otro inmovilizado mat...,216,Mobiliario,21600004,4 ARCHIVADORES VERTICALES,1460.00
20,2025,Anual,ACTIVO,Activo No Corriente,II,Inmovilizado material,2,Instalaciones técnicas y otro inmovilizado mat...,216,Mobiliario,21600005,6 ARCHIVADORES METALICOS,1507.00
21,2025,Anual,ACTIVO,Activo No Corriente,II,Inmovilizado material,2,Instalaciones técnicas y otro inmovilizado mat...,216,Mobiliario,21600006,2 ARMARIOS ALTOS DE PUERTAS,700.00
22,2025,Anual,ACTIVO,Activo No Corriente,II,Inmovilizado material,2,Instalaciones técnicas y otro inmovilizado mat...,216,Mobiliario,21600007,4 PUF REDONDO O CUADRADO,1112.49
23,2025,Anual,ACTIVO,Activo No Corriente,II,Inmovilizado material,2,Instalaciones técnicas y otro inmovilizado mat...,216,Mobiliario,21600008,MESA DE JUNTAS DE MEDIDAS 200X120 SOBRE LAMINA...,990.80


### Resultado y Descarga: 

In [242]:
Balance.head(4)

,Año,Periodo,Partida,Subpartida,Grupo_id,Grupo,SubGrupo_id,SubGrupo,Cuenta_id,Cuenta,Subcuenta_id,Subcuenta,Valor
10,2025,Anual,ACTIVO,Activo No Corriente,I,Inmovilizado intangible,5,Aplicaciones informáticas,206,Aplicaciones informáticas,20600001,CASH LESS DESARROLLO SOFTWARE,718791.00
12,2025,Anual,ACTIVO,Activo No Corriente,I,Inmovilizado intangible,5,Aplicaciones informáticas,2806,Amortización acumulada de aplicaciones informá...,28060001,AMORTACUMCASH LESS,-663256.75
16,2025,Anual,ACTIVO,Activo No Corriente,II,Inmovilizado material,2,Instalaciones técnicas y otro inmovilizado mat...,216,Mobiliario,21600001,5 ASIENTOS RECLINABLES,1650.00
17,2025,Anual,ACTIVO,Activo No Corriente,II,Inmovilizado material,2,Instalaciones técnicas y otro inmovilizado mat...,216,Mobiliario,21600002,5 MESAS OFICINA LINEAL,1061.00


In [243]:
### Descarga arhivo: 
NombreArchivo = f"File.Balance{Año}.csv"
Balance.to_csv(NombreArchivo, index=False)

In [244]:
f"Archivo creado llamado: 'File.Balance{Año}.csv'"

"Archivo creado llamado: 'File.Balance2025.csv'"